In [15]:
import rdflib
from rdflib import Graph
from rdflib import URIRef

In [7]:
niosh = Graph()
niosh.parse('./mappings/niosh_rdf_tutV2c.ttl')

<Graph identifier=Nddd7b4e60fbe4a44a6eec92678b3fa0f (<class 'rdflib.graph.Graph'>)>

In [3]:
def get_graph_summary(graph):
    """Generate a summary of the graph contents."""
    summary = {}
    
    # Count triples
    summary["total_triples"] = len(graph)
    
    # Count subjects, predicates, objects
    subjects = set(graph.subjects())
    predicates = set(graph.predicates())
    objects = set(graph.objects())
    
    summary["unique_subjects"] = len(subjects)
    summary["unique_predicates"] = len(predicates)
    summary["unique_objects"] = len(objects)
    
    # Count types of resources
    classes = {}
    for s, o in graph.subject_objects(RDF.type):
        class_name = str(o).split("/")[-1].split("#")[-1]
        classes[class_name] = classes.get(class_name, 0) + 1
    
    summary["resource_types"] = classes
    
    # Find common predicates
    pred_counts = {}
    for p in predicates:
        pred_name = str(p).split("/")[-1].split("#")[-1]
        count = len(list(graph.subject_objects(p)))
        pred_counts[pred_name] = count
    
    summary["predicate_frequencies"] = dict(sorted(pred_counts.items(), key=lambda x: x[1], reverse=True)[:10])
    
    return summary

def print_graph_summary(summary):
    """Print a user-friendly summary of the graph."""
    print("\n=== RDF GRAPH SUMMARY ===")
    print(f"Total triples: {summary['total_triples']}")
    print(f"Unique subjects: {summary['unique_subjects']}")
    print(f"Unique predicates: {summary['unique_predicates']}")
    print(f"Unique objects: {summary['unique_objects']}")
    
    print("\nResource types:")
    for class_name, count in sorted(summary["resource_types"].items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  - {class_name}: {count} instances")
    
    print("\nMost common predicates:")
    for pred_name, count in summary["predicate_frequencies"].items():
        print(f"  - {pred_name}: {count} occurrences")

In [4]:
results = niosh.query("""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>

    SELECT ?assayLabel (COUNT(?assay) as ?count)
    WHERE {
    ?assay a obo:OBI_0000070 ;
            rdfs:label ?assayLabel .
    }
    GROUP BY ?assayLabel
    ORDER BY DESC(?count)
""")

In [5]:
get_graph_summary(niosh)

NameError: name 'RDF' is not defined

In [8]:
def find_central_entities(graph, top_n=10):
    """Find entities that have the most connections."""
    entity_connections = {}
    
    # Count outgoing connections
    for s in set(graph.subjects()):
        entity_connections[s] = entity_connections.get(s, 0) + len(list(graph.predicate_objects(s)))
    
    # Count incoming connections
    for o in set(graph.objects()):
        if isinstance(o, (str)) or not o.startswith('_:'):  # Skip blank nodes
            entity_connections[o] = entity_connections.get(o, 0) + len(list(graph.subject_predicates(o)))
    
    # Sort and return top entities
    sorted_entities = sorted(entity_connections.items(), key=lambda x: x[1], reverse=True)
    return sorted_entities[:top_n]

def extract_entity_details(graph, entity_uri):
    """Extract all properties for a given entity."""
    details = {}
    
    # Get all predicate-object pairs for this entity
    for p, o in graph.predicate_objects(entity_uri):
        pred_name = str(p).split("/")[-1].split("#")[-1]
        
        # Format the object value
        if isinstance(o, (str)):
            obj_value = o
        else:
            # For URIs, just take the last part
            obj_value = str(o).split("/")[-1].split("#")[-1]
        
        details[pred_name] = obj_value
    
    return details

In [9]:
find_central_entities(niosh)

[(rdflib.term.Literal('nan', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#double')),
  5156),
 (rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C53321'),
  797),
 (rdflib.term.URIRef('http://www.ebi.ac.uk/efo/EFO_0000487'), 431),
 (rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C25488'),
  431),
 (rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C68568'),
  431),
 (rdflib.term.URIRef('https://w3id.org/biolink/vocab/SubjectOfInvestigation'),
  431),
 (rdflib.term.URIRef('http://purl.obolibrary.org/obo/OBI_0000070'), 345),
 (rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C12919'),
  345),
 (rdflib.term.URIRef('http://www.bioassayontology.org/bao#BAO_0000179'), 345),
 (rdflib.term.Literal('Lung'), 344)]

In [10]:
extract_entity_details(niosh, 'http://purl.obolibrary.org/obo/OBI_0000070')

{}

In [11]:
print("Found {} results:".format(len(results)))
for row in results:
    print(f"Assay: {row.assay}, Label: {row.assayLabel}")

Found 6 results:


AttributeError: assay

In [12]:
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, RDFS

# Create namespaces
ASSAY = Namespace("http://example.org/assays/row/")
BAO = Namespace("http://www.bioassayontology.org/bao#")
DCTERMS = Namespace("http://purl.org/dc/terms/")
OBO = Namespace("http://purl.obolibrary.org/obo/")

# Load your RDF data
# g = Graph()
# g.parse("your_data.ttl", format="turtle")

# Run a SPARQL query
query = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX obo: <http://purl.obolibrary.org/obo/>

SELECT ?assay ?assayLabel
WHERE {
  ?assay a obo:OBI_0000070 ;
         rdfs:label ?assayLabel .
}
LIMIT 10
"""

results = niosh.query(query)

# Method 1: Access by index position
print("Found {} results:".format(len(results)))
for row in results:
    print(f"Assay: {row[0]}, Label: {row[1]}")

# Method 2: Access by variable name
print("\nUsing variable names:")
for row in results:
    print(f"Assay: {row['assay']}, Label: {row['assayLabel']}")

# Method 3: Convert to dictionary first
print("\nConverting to dictionary:")
for row in results:
    row_dict = row.asdict()
    print(f"Assay: {row_dict['assay']}, Label: {row_dict['assayLabel']}")

# For more complex formatting:
print("\nFormatted results table:")
print("{:<50} | {:<20}".format("Assay URI", "Label"))
print("-" * 75)
for row in results:
    print("{:<50} | {:<20}".format(str(row[0]), str(row[1])))

Found 10 results:
Assay: http://example.org/assays/row/0, Label: LDH
Assay: http://example.org/assays/row/1, Label: LDH
Assay: http://example.org/assays/row/10, Label: LDH
Assay: http://example.org/assays/row/100, Label: Cytokine level/production
Assay: http://example.org/assays/row/101, Label: Cytokine level/production
Assay: http://example.org/assays/row/102, Label: Cytokine level/production
Assay: http://example.org/assays/row/103, Label: Cytokine level/production
Assay: http://example.org/assays/row/104, Label: Cytokine level/production
Assay: http://example.org/assays/row/105, Label: Cytokine level/production
Assay: http://example.org/assays/row/106, Label: Cytokine level/production

Using variable names:
Assay: http://example.org/assays/row/0, Label: LDH
Assay: http://example.org/assays/row/1, Label: LDH
Assay: http://example.org/assays/row/10, Label: LDH
Assay: http://example.org/assays/row/100, Label: Cytokine level/production
Assay: http://example.org/assays/row/101, Label: Cy

In [ ]:
results.vars

In [ ]:
pip install pyvis

In [ ]:
pip install numpy==1.24.3

In [13]:
from rdflib import Graph, RDF, RDFS, OWL
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network

def extract_rdf_schema(graph):
    """Extract schema information from an RDF graph."""
    schema_info = {
        'classes': [],
        'properties': [],
        'class_relationships': [],
        'property_relationships': []
    }
    
    # Find all classes
    for s, p, o in graph.triples((None, RDF.type, RDFS.Class)):
        schema_info['classes'].append((s, 'Class'))
    
    for s, p, o in graph.triples((None, RDF.type, OWL.Class)):
        schema_info['classes'].append((s, 'Class'))
    
    # Find all properties
    for s, p, o in graph.triples((None, RDF.type, RDF.Property)):
        schema_info['properties'].append((s, 'Property'))
    
    for s, p, o in graph.triples((None, RDF.type, OWL.DatatypeProperty)):
        schema_info['properties'].append((s, 'DatatypeProperty'))
    
    for s, p, o in graph.triples((None, RDF.type, OWL.ObjectProperty)):
        schema_info['properties'].append((s, 'ObjectProperty'))
    
    # Find class hierarchies
    for s, p, o in graph.triples((None, RDFS.subClassOf, None)):
        schema_info['class_relationships'].append((s, 'subClassOf', o))
    
    # Find property domains and ranges
    for s, p, o in graph.triples((None, RDFS.domain, None)):
        schema_info['property_relationships'].append((s, 'domain', o))
    
    for s, p, o in graph.triples((None, RDFS.range, None)):
        schema_info['property_relationships'].append((s, 'range', o))
    
    return schema_info

def create_schema_visualization(schema_info, output_file="rdf_schema.html"):
    """Create a visualization of the RDF schema."""
    net = Network(height="800px", width="100%", directed=True, notebook=False)
    
    # Add class nodes
    for class_uri, class_type in schema_info['classes']:
        label = str(class_uri).split('/')[-1].split('#')[-1]
        net.add_node(str(class_uri), label=label, title=str(class_uri), 
                    color="#8dd3c7", shape="box", group=1)
    
    # Add property nodes
    for prop_uri, prop_type in schema_info['properties']:
        label = str(prop_uri).split('/')[-1].split('#')[-1]
        if prop_type == 'DatatypeProperty':
            color = "#fb8072"  # Red for datatype properties
        else:
            color = "#80b1d3"  # Blue for object properties
        
        net.add_node(str(prop_uri), label=label, title=str(prop_uri),
                    color=color, shape="ellipse", group=2)
    
    # Add class relationships
    for s, rel_type, o in schema_info['class_relationships']:
        net.add_edge(str(s), str(o), label="subClassOf", arrows="to")
    
    # Add property relationships
    for prop, rel_type, target in schema_info['property_relationships']:
        if rel_type == 'domain':
            net.add_edge(str(target), str(prop), label="has property", arrows="to", 
                        color="#888888", dashes=True)
        elif rel_type == 'range':
            net.add_edge(str(prop), str(target), label="has range", arrows="to",
                        color="#888888", dashes=True)
    
    # Configure the visualization
    net.show_buttons(['physics'])
    net.barnes_hut(spring_length=200)
    
    # Save the visualization
    net.save_graph(output_file)
    print(f"Schema visualization saved to {output_file}")
    return output_file

def infer_schema_from_instances(graph):
    """Infer schema information from instance data."""
    inferred_schema = {
        'classes': set(),
        'properties': set(),
        'property_domains': {},
        'property_ranges': {}
    }
    
    # Find all types used in the graph
    for s, p, o in graph.triples((None, RDF.type, None)):
        inferred_schema['classes'].add(o)
    
    # Find all predicates used in the graph
    for s, p, o in graph:
        if p != RDF.type:
            inferred_schema['properties'].add(p)
            
            # Track which types use this property (domain)
            for _, _, type_o in graph.triples((s, RDF.type, None)):
                if p not in inferred_schema['property_domains']:
                    inferred_schema['property_domains'][p] = set()
                inferred_schema['property_domains'][p].add(type_o)
            
            # Track value types for this property (range)
            if isinstance(o, URIRef):
                for _, _, type_o in graph.triples((o, RDF.type, None)):
                    if p not in inferred_schema['property_ranges']:
                        inferred_schema['property_ranges'][p] = set()
                    inferred_schema['property_ranges'][p].add(type_o)
    
    # Convert to format usable by visualization function
    schema_info = {
        'classes': [(c, 'InferredClass') for c in inferred_schema['classes']],
        'properties': [(p, 'InferredProperty') for p in inferred_schema['properties']],
        'class_relationships': [],
        'property_relationships': []
    }
    
    # Add domain and range relationships
    for p, domains in inferred_schema['property_domains'].items():
        for domain in domains:
            schema_info['property_relationships'].append((p, 'domain', domain))
    
    for p, ranges in inferred_schema['property_ranges'].items():
        for range_val in ranges:
            schema_info['property_relationships'].append((p, 'range', range_val))
    
    return schema_info

def generate_schema_diagram(graph, output_file="rdf_schema.html", infer_from_instances=True):
    """Generate a complete RDF schema diagram."""
    # First try to extract explicit schema information
    schema_info = extract_rdf_schema(graph)
    
    # If requested, also infer schema from instances
    if infer_from_instances:
        inferred_schema = infer_schema_from_instances(graph)
        
        # Merge explicit and inferred schema information
        for category in ['classes', 'properties', 'class_relationships', 'property_relationships']:
            schema_info[category].extend(inferred_schema[category])
    
    # Create visualization
    return create_schema_visualization(schema_info, output_file)

In [16]:
generate_schema_diagram(niosh)

Schema visualization saved to rdf_schema.html


'rdf_schema.html'

In [17]:
from rdflib import Graph, RDF, RDFS, OWL, URIRef
from pyvis.network import Network
import os
import pandas as pd

def extract_schema_without_matplotlib(graph, output_dir="rdf_schema"):
    """Extract and visualize RDF schema without using matplotlib."""
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Step 1: Extract schema information
    schema_info = {
        'classes': set(),
        'properties': set(),
        'class_hierarchy': [],
        'property_domains': {},
        'property_ranges': {}
    }
    
    # Find explicit schema information
    for s, p, o in graph.triples((None, RDF.type, RDFS.Class)):
        schema_info['classes'].add(s)
    
    for s, p, o in graph.triples((None, RDF.type, OWL.Class)):
        schema_info['classes'].add(s)
    
    for s, p, o in graph.triples((None, RDF.type, RDF.Property)):
        schema_info['properties'].add(s)
    
    for s, p, o in graph.triples((None, RDF.type, OWL.DatatypeProperty)):
        schema_info['properties'].add(s)
    
    for s, p, o in graph.triples((None, RDF.type, OWL.ObjectProperty)):
        schema_info['properties'].add(s)
    
    # Class hierarchy
    for s, p, o in graph.triples((None, RDFS.subClassOf, None)):
        schema_info['class_hierarchy'].append((s, o))
    
    # Property domains and ranges
    for s, p, o in graph.triples((None, RDFS.domain, None)):
        if s not in schema_info['property_domains']:
            schema_info['property_domains'][s] = []
        schema_info['property_domains'][s].append(o)
    
    for s, p, o in graph.triples((None, RDFS.range, None)):
        if s not in schema_info['property_ranges']:
            schema_info['property_ranges'][s] = []
        schema_info['property_ranges'][s].append(o)
    
    # If no explicit schema is found, infer from instance data
    if not schema_info['classes']:
        print("No explicit schema found, inferring from instance data...")
        
        # Find all types used in the graph
        for s, p, o in graph.triples((None, RDF.type, None)):
            schema_info['classes'].add(o)
        
        # Find all predicates used in the graph
        for s, p, o in graph:
            if p not in (RDF.type, RDFS.subClassOf, RDFS.domain, RDFS.range):
                schema_info['properties'].add(p)
                
                # Infer domain from usage
                types = list(graph.objects(s, RDF.type))
                if types:
                    if p not in schema_info['property_domains']:
                        schema_info['property_domains'][p] = []
                    for t in types:
                        if t not in schema_info['property_domains'][p]:
                            schema_info['property_domains'][p].append(t)
                
                # Infer range if object is a URI with a type
                if isinstance(o, URIRef):
                    types = list(graph.objects(o, RDF.type))
                    if types:
                        if p not in schema_info['property_ranges']:
                            schema_info['property_ranges'][p] = []
                        for t in types:
                            if t not in schema_info['property_ranges'][p]:
                                schema_info['property_ranges'][p].append(t)
    
    # Step 2: Create a visual representation with pyvis
    net = Network(height="800px", width="100%", directed=True, notebook=False)
    
    # Add class nodes
    for class_uri in schema_info['classes']:
        label = str(class_uri).split('/')[-1].split('#')[-1]
        net.add_node(str(class_uri), label=label, title=str(class_uri), 
                    color="#8dd3c7", shape="box", group=1)
    
    # Add property nodes
    for prop_uri in schema_info['properties']:
        label = str(prop_uri).split('/')[-1].split('#')[-1]
        net.add_node(str(prop_uri), label=label, title=str(prop_uri),
                    color="#80b1d3", shape="ellipse", group=2)
    
    # Add class hierarchy
    for subclass, superclass in schema_info['class_hierarchy']:
        if subclass in schema_info['classes'] and superclass in schema_info['classes']:
            net.add_edge(str(subclass), str(superclass), label="subClassOf", arrows="to")
    
    # Add property domain/range relationships
    for prop, domains in schema_info['property_domains'].items():
        for domain in domains:
            if domain in schema_info['classes'] and prop in schema_info['properties']:
                net.add_edge(str(domain), str(prop), label="has property", arrows="to", 
                            color="#888888", dashes=True)
    
    for prop, ranges in schema_info['property_ranges'].items():
        for range_val in ranges:
            if range_val in schema_info['classes'] and prop in schema_info['properties']:
                net.add_edge(str(prop), str(range_val), label="has range", arrows="to",
                            color="#888888", dashes=True)
    
    # Configure visualization
    net.barnes_hut(spring_length=200)
    net.show_buttons(['physics'])
    
    # Save the visualization
    output_file = os.path.join(output_dir, "rdf_schema.html")
    net.save_graph(output_file)
    print(f"Schema visualization saved to {output_file}")
    
    # Step 3: Create tabular representations for easier understanding
    # Classes table
    classes_data = []
    for class_uri in schema_info['classes']:
        label = str(class_uri).split('/')[-1].split('#')[-1]
        
        # Count instances of this class
        instances_count = len(list(graph.subjects(RDF.type, class_uri)))
        
        # Find subclasses
        subclasses = [str(s).split('/')[-1].split('#')[-1] 
                     for s, p, o in graph.triples((None, RDFS.subClassOf, class_uri))]
        
        # Find properties that have this class as domain
        properties = []
        for prop, domains in schema_info['property_domains'].items():
            if class_uri in domains:
                properties.append(str(prop).split('/')[-1].split('#')[-1])
        
        classes_data.append({
            'Class': label,
            'URI': str(class_uri),
            'Instances': instances_count,
            'Subclasses': ', '.join(subclasses) if subclasses else '-',
            'Properties': ', '.join(properties) if properties else '-'
        })
    
    classes_df = pd.DataFrame(classes_data)
    classes_csv = os.path.join(output_dir, "classes.csv")
    classes_df.to_csv(classes_csv, index=False)
    
    # Properties table
    properties_data = []
    for prop_uri in schema_info['properties']:
        label = str(prop_uri).split('/')[-1].split('#')[-1]
        
        # Get domains and ranges
        domains = [str(d).split('/')[-1].split('#')[-1] 
                  for d in schema_info['property_domains'].get(prop_uri, [])]
        
        ranges = [str(r).split('/')[-1].split('#')[-1] 
                 for r in schema_info['property_ranges'].get(prop_uri, [])]
        
        # Count usage of this property
        usage_count = len(list(graph.subject_objects(prop_uri)))
        
        properties_data.append({
            'Property': label,
            'URI': str(prop_uri),
            'Domains': ', '.join(domains) if domains else '-',
            'Ranges': ', '.join(ranges) if ranges else '-',
            'Usage Count': usage_count
        })
    
    properties_df = pd.DataFrame(properties_data)
    properties_csv = os.path.join(output_dir, "properties.csv")
    properties_df.to_csv(properties_csv, index=False)
    
    # Step 4: Create an HTML index file that links everything together
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>RDF Schema Visualization</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; line-height: 1.6; }}
            h1, h2, h3 {{ color: #2c3e50; }}
            table {{ border-collapse: collapse; width: 100%; margin-bottom: 20px; }}
            th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
            th {{ background-color: #f2f2f2; font-weight: bold; }}
            tr:nth-child(even) {{ background-color: #f9f9f9; }}
            .section {{ margin-bottom: 30px; }}
            .visualization-link {{ display: inline-block; margin-top: 10px; padding: 10px 15px; 
                                 background-color: #3498db; color: white; text-decoration: none; 
                                 border-radius: 4px; }}
            .visualization-link:hover {{ background-color: #2980b9; }}
            .legend {{ display: flex; margin: 20px 0; }}
            .legend-item {{ display: flex; align-items: center; margin-right: 20px; }}
            .legend-color {{ width: 20px; height: 20px; margin-right: 5px; }}
            .download-link {{ color: #3498db; text-decoration: none; }}
            .download-link:hover {{ text-decoration: underline; }}
        </style>
    </head>
    <body>
        <h1>RDF Schema Visualization</h1>
        
        <div class="section">
            <h2>Schema Overview</h2>
            <p>This visualization represents the schema extracted from your RDF data.</p>
            
            <div class="legend">
                <div class="legend-item">
                    <div class="legend-color" style="background-color: #8dd3c7;"></div>
                    <span>Classes</span>
                </div>
                <div class="legend-item">
                    <div class="legend-color" style="background-color: #80b1d3;"></div>
                    <span>Properties</span>
                </div>
            </div>
            
            <p>Key statistics:</p>
            <ul>
                <li><strong>Classes:</strong> {len(schema_info['classes'])}</li>
                <li><strong>Properties:</strong> {len(schema_info['properties'])}</li>
                <li><strong>Class hierarchy relationships:</strong> {len(schema_info['class_hierarchy'])}</li>
                <li><strong>Property domain definitions:</strong> {len(schema_info['property_domains'])}</li>
                <li><strong>Property range definitions:</strong> {len(schema_info['property_ranges'])}</li>
            </ul>
            
            <a href="rdf_schema.html" target="_blank" class="visualization-link">Open Interactive Schema Visualization</a>
        </div>
        
        <div class="section">
            <h2>Classes</h2>
            <p>The following classes were identified in the schema:</p>
            
            {classes_df.to_html(index=False)}
            
            <p><a href="classes.csv" class="download-link">Download as CSV</a></p>
        </div>
        
        <div class="section">
            <h2>Properties</h2>
            <p>The following properties were identified in the schema:</p>
            
            {properties_df.to_html(index=False)}
            
            <p><a href="properties.csv" class="download-link">Download as CSV</a></p>
        </div>
        
        <div class="section">
            <h2>How to Read This Schema</h2>
            <p>The interactive visualization shows the relationships between classes and properties in your RDF data:</p>
            <ul>
                <li><strong>Boxes (green):</strong> Represent classes</li>
                <li><strong>Circles (blue):</strong> Represent properties</li>
                <li><strong>Solid arrows:</strong> Indicate class hierarchy relationships (subClassOf)</li>
                <li><strong>Dashed arrows from class to property:</strong> Indicate that the property has that class as its domain</li>
                <li><strong>Dashed arrows from property to class:</strong> Indicate that the property has that class as its range</li>
            </ul>
            <p>You can:</p>
            <ul>
                <li>Drag nodes to rearrange the visualization</li>
                <li>Zoom in/out using mouse wheel</li>
                <li>Click on a node to highlight its connections</li>
                <li>Use the physics controls to adjust the layout</li>
            </ul>
        </div>
    </body>
    </html>
    """
    
    index_file = os.path.join(output_dir, "index.html")
    with open(index_file, "w") as f:
        f.write(html_content)
    
    print(f"Complete schema documentation created at {index_file}")
    return {
        'visualization': output_file,
        'index': index_file,
        'classes': classes_csv,
        'properties': properties_csv
    }

# Example usage
def visualize_rdf_schema(file_path, output_dir="rdf_schema"):
    """Generate an RDF schema visualization from an RDF file."""
    # Load the graph
    graph = Graph()
    graph.parse(file_path)
    print(f"Loaded {len(graph)} triples from {file_path}")
    
    # Extract schema and create visualizations
    return extract_schema_without_matplotlib(graph, output_dir)

/Users/pranavsingh/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [20]:
visualize_rdf_schema('./mappings/niosh_rdf_tutV2c.ttl', output_dir='niosh')

Loaded 20304 triples from ./mappings/niosh_rdf_tutV2c.ttl
No explicit schema found, inferring from instance data...
Schema visualization saved to niosh/rdf_schema.html
Complete schema documentation created at niosh/index.html


{'visualization': 'niosh/rdf_schema.html',
 'index': 'niosh/index.html',
 'classes': 'niosh/classes.csv',
 'properties': 'niosh/properties.csv'}

In [21]:
visualize_rdf_schema('./mappings/NKB_RDF_V3.ttl', output_dir='nkb')

Loaded 1369675 triples from ./mappings/NKB_RDF_V3.ttl
No explicit schema found, inferring from instance data...
Schema visualization saved to nkb/rdf_schema.html
Complete schema documentation created at nkb/index.html


{'visualization': 'nkb/rdf_schema.html',
 'index': 'nkb/index.html',
 'classes': 'nkb/classes.csv',
 'properties': 'nkb/properties.csv'}

In [22]:
visualize_rdf_schema('./mappings/cpsc_database.ttl', output_dir='cpsc')

Loaded 29596 triples from ./mappings/cpsc_database.ttl
No explicit schema found, inferring from instance data...
Schema visualization saved to cpsc/rdf_schema.html
Complete schema documentation created at cpsc/index.html


{'visualization': 'cpsc/rdf_schema.html',
 'index': 'cpsc/index.html',
 'classes': 'cpsc/classes.csv',
 'properties': 'cpsc/properties.csv'}

In [24]:
from rdflib import Graph, RDF, RDFS, OWL, URIRef, BNode, Literal
from collections import defaultdict
import os
import json

def identify_and_classify_resources(graph):
    """Identify and classify all resources in the graph including blank nodes."""
    classification = {
        'classes': set(),
        'properties': set(),
        'bnodes': set(),
        'nodebags': {},
        'primary_entities': set(),
        'secondary_entities': set(),
        'link_entities': set(),
        'property_tables': {}
    }
    
    # Find all classes
    for s, p, o in graph.triples((None, RDF.type, RDFS.Class)):
        classification['classes'].add(s)
    
    for s, p, o in graph.triples((None, RDF.type, OWL.Class)):
        classification['classes'].add(s)
    
    # If no explicit classes, find all types used in instance data
    if not classification['classes']:
        for s, p, o in graph.triples((None, RDF.type, None)):
            classification['classes'].add(o)
    
    # Find all properties
    for s, p, o in graph.triples((None, RDF.type, RDF.Property)):
        classification['properties'].add(s)
    
    for s, p, o in graph.triples((None, RDF.type, OWL.DatatypeProperty)):
        classification['properties'].add(s)
    
    for s, p, o in graph.triples((None, RDF.type, OWL.ObjectProperty)):
        classification['properties'].add(s)
    
    # If no explicit properties, find all predicates used
    if not classification['properties']:
        for s, p, o in graph:
            if p != RDF.type:
                classification['properties'].add(p)
    
    # Identify blank nodes
    for s in graph.subjects():
        if isinstance(s, BNode):
            classification['bnodes'].add(s)
    
    for o in graph.objects():
        if isinstance(o, BNode):
            classification['bnodes'].add(o)
    
    # Analyze blank nodes to identify nodebags
    for bnode in classification['bnodes']:
        # Count outgoing properties
        outgoing_props = list(graph.predicate_objects(bnode))
        
        # Count incoming references
        incoming_refs = list(graph.subject_predicates(bnode))
        
        # A nodebag typically has multiple outgoing properties and at least one incoming reference
        if len(outgoing_props) >= 2 and len(incoming_refs) >= 1:
            # Create a nodebag entry
            bag_props = {}
            for p, o in outgoing_props:
                pred_name = str(p).split('/')[-1].split('#')[-1]
                
                if isinstance(o, Literal):
                    obj_value = str(o)
                else:
                    obj_value = str(o).split('/')[-1].split('#')[-1]
                
                bag_props[pred_name] = obj_value
            
            # Record references to this nodebag
            refs = {}
            for s, p in incoming_refs:
                pred_name = str(p).split('/')[-1].split('#')[-1]
                
                if isinstance(s, BNode):
                    subj_value = f"bnode:{str(s)}"
                else:
                    subj_value = str(s).split('/')[-1].split('#')[-1]
                
                refs[pred_name] = subj_value
            
            classification['nodebags'][str(bnode)] = {
                'properties': bag_props,
                'references': refs
            }
    
    # Identify primary and secondary entities based on connectivity pattern
    entity_links = defaultdict(int)
    for s, p, o in graph:
        if isinstance(s, URIRef) and isinstance(o, URIRef) and s != o:
            entity_links[s] += 1
            entity_links[o] += 1
    
    # Entities with high connectivity are likely primary entities
    threshold = sum(entity_links.values()) / len(entity_links) if entity_links else 0
    
    for entity, count in entity_links.items():
        if count > threshold:
            classification['primary_entities'].add(entity)
        else:
            classification['secondary_entities'].add(entity)
    
    # Identify link entities (entities that primarily connect other entities)
    for entity in classification['secondary_entities'].copy():
        outgoing_links = 0
        for p, o in graph.predicate_objects(entity):
            if isinstance(o, URIRef) and o in classification['primary_entities']:
                outgoing_links += 1
        
        if outgoing_links >= 2:
            classification['link_entities'].add(entity)
            classification['secondary_entities'].remove(entity)
    
    # Identify property tables (sets of properties that always occur together)
    property_patterns = defaultdict(set)
    
    # For each subject, record its property set
    for s in set(graph.subjects()):
        if not isinstance(s, BNode):  # Skip blank nodes
            props = frozenset(p for p, o in graph.predicate_objects(s))
            if len(props) > 1:  # Only consider entities with multiple properties
                property_patterns[props].add(s)
    
    # Keep patterns that occur multiple times
    for pattern, entities in property_patterns.items():
        if len(entities) >= 2:
            props_list = sorted([str(p).split('/')[-1].split('#')[-1] for p in pattern])
            pattern_name = f"PropertyGroup_{'_'.join(props_list[:3])}"
            if len(props_list) > 3:
                pattern_name += f"_and_{len(props_list) - 3}_more"
            
            classification['property_tables'][pattern_name] = {
                'properties': [str(p) for p in pattern],
                'entities': [str(e) for e in entities]
            }
    
    return classification

def generate_schema_report(graph, output_dir="complex_rdf_schema"):
    """Generate a comprehensive schema report for a complex RDF graph."""
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Analyze the graph structure
    classification = identify_and_classify_resources(graph)
    
    # Create text report
    text_report = os.path.join(output_dir, "schema_report.txt")
    with open(text_report, 'w') as f:
        f.write("COMPLEX RDF SCHEMA ANALYSIS\n")
        f.write("==========================\n\n")
        
        f.write(f"Total triples: {len(graph)}\n")
        f.write(f"Classes: {len(classification['classes'])}\n")
        f.write(f"Properties: {len(classification['properties'])}\n")
        f.write(f"Blank nodes: {len(classification['bnodes'])}\n")
        f.write(f"Nodebags: {len(classification['nodebags'])}\n")
        f.write(f"Primary entities: {len(classification['primary_entities'])}\n")
        f.write(f"Secondary entities: {len(classification['secondary_entities'])}\n")
        f.write(f"Link entities: {len(classification['link_entities'])}\n")
        f.write(f"Property table patterns: {len(classification['property_tables'])}\n\n")
        
        f.write("CLASSES\n")
        f.write("=======\n")
        for c in sorted([str(c) for c in classification['classes']]):
            class_name = c.split('/')[-1].split('#')[-1]
            instances = len(list(graph.subjects(RDF.type, URIRef(c))))
            f.write(f"* {class_name} ({c}) - {instances} instances\n")
        
        f.write("\nNODEBAGS\n")
        f.write("========\n")
        for bnode_id, bag_info in classification['nodebags'].items():
            f.write(f"* Nodebag: {bnode_id}\n")
            f.write(f"  Referenced by: {bag_info['references']}\n")
            f.write(f"  Properties: {bag_info['properties']}\n\n")
        
        f.write("\nPROPERTY TABLE PATTERNS\n")
        f.write("======================\n")
        for pattern_name, pattern_info in classification['property_tables'].items():
            f.write(f"* Pattern: {pattern_name}\n")
            f.write(f"  Properties: {[p.split('/')[-1].split('#')[-1] for p in pattern_info['properties']]}\n")
            f.write(f"  Used by {len(pattern_info['entities'])} entities\n\n")
    
    # Create JSON data file
    json_file = os.path.join(output_dir, "schema_data.json")
    with open(json_file, 'w') as f:
        # Convert sets to lists for JSON serialization
        serializable = {
            'classes': [str(c) for c in classification['classes']],
            'properties': [str(p) for p in classification['properties']],
            'bnodes': [str(b) for b in classification['bnodes']],
            'nodebags': classification['nodebags'],
            'primary_entities': [str(e) for e in classification['primary_entities']],
            'secondary_entities': [str(e) for e in classification['secondary_entities']],
            'link_entities': [str(e) for e in classification['link_entities']],
            'property_tables': classification['property_tables']
        }
        json.dump(serializable, f, indent=2)
    
    # Create HTML report - FIXED to avoid f-string with # issues
    html_report = os.path.join(output_dir, "schema_report.html")
    with open(html_report, 'w') as f:
        # Start of HTML
        html_content = """
        <!DOCTYPE html>
        <html>
        <head>
            <title>Complex RDF Schema Analysis</title>
            <style>
                body { font-family: Arial, sans-serif; margin: 20px; line-height: 1.6; }
                h1, h2, h3 { color: #2c3e50; }
                table { border-collapse: collapse; width: 100%; margin-bottom: 20px; }
                th, td { border: 1px solid #ddd; padding: 8px; text-align: left; }
                th { background-color: #f2f2f2; font-weight: bold; }
                tr:nth-child(even) { background-color: #f9f9f9; }
                .section { margin-bottom: 30px; }
                pre { background-color: #f8f8f8; padding: 10px; border-radius: 5px; overflow-x: auto; }
                .note { background-color: #ffffcc; padding: 10px; border-left: 4px solid #ffeb3b; }
                .card { border: 1px solid #ddd; border-radius: 5px; padding: 15px; margin-bottom: 15px; }
                .card-title { font-weight: bold; margin-bottom: 10px; }
                .badge { display: inline-block; padding: 3px 8px; background-color: #3498db; color: white; border-radius: 10px; font-size: 12px; margin-right: 5px; }
            </style>
        </head>
        <body>
            <h1>Complex RDF Schema Analysis</h1>
            
            <div class="section">
                <h2>Overview</h2>
                <table>
                    <tr><th>Metric</th><th>Count</th></tr>
        """
        
        # Add overview metrics
        html_content += f"""
                    <tr><td>Total Triples</td><td>{len(graph)}</td></tr>
                    <tr><td>Classes</td><td>{len(classification['classes'])}</td></tr>
                    <tr><td>Properties</td><td>{len(classification['properties'])}</td></tr>
                    <tr><td>Blank Nodes</td><td>{len(classification['bnodes'])}</td></tr>
                    <tr><td>Nodebags</td><td>{len(classification['nodebags'])}</td></tr>
                    <tr><td>Primary Entities</td><td>{len(classification['primary_entities'])}</td></tr>
                    <tr><td>Secondary Entities</td><td>{len(classification['secondary_entities'])}</td></tr>
                    <tr><td>Link Entities</td><td>{len(classification['link_entities'])}</td></tr>
                    <tr><td>Property Table Patterns</td><td>{len(classification['property_tables'])}</td></tr>
                </table>
            </div>
            
            <div class="section">
                <h2>Classes</h2>
                <table>
                    <tr><th>Class</th><th>URI</th><th>Instances</th></tr>
        """
        
        # Add classes - avoiding f-string with # issue
        for c in sorted([str(c) for c in classification['classes']]):
            class_name = c.split('/')[-1].split('#')[-1]
            instances = len(list(graph.subjects(RDF.type, URIRef(c))))
            html_content += f"""
                    <tr>
                        <td>{class_name}</td>
                        <td>{c}</td>
                        <td>{instances}</td>
                    </tr>
            """
        
        html_content += """
                </table>
            </div>
            
            <div class="section">
                <h2>Nodebags</h2>
                <p>Nodebags are blank nodes that group related properties together:</p>
        """
        
        # Add nodebags - avoiding f-string with # issue
        nodebag_items = list(classification['nodebags'].items())
        display_nodebags = nodebag_items[:10]  # Limit to first 10 for brevity
        
        for bnode_id, bag_info in display_nodebags:
            html_content += f"""
                <div class="card">
                    <div class="card-title">Nodebag: {bnode_id}</div>
                    <p><strong>Referenced by:</strong> 
            """
            
            # Add references
            for ref_type, ref_val in bag_info['references'].items():
                html_content += f'<span class="badge">{ref_type}: {ref_val}</span>'
            
            html_content += """
                    </p>
                    <p><strong>Properties:</strong></p>
                    <table>
                        <tr><th>Property</th><th>Value</th></tr>
            """
            
            # Add properties
            for prop, val in bag_info['properties'].items():
                html_content += f"""
                        <tr>
                            <td>{prop}</td>
                            <td>{val}</td>
                        </tr>
                """
            
            html_content += """
                    </table>
                </div>
            """
        
        # Note if showing limited nodebags
        if len(classification['nodebags']) > 10:
            html_content += f'<p><em>Note: Only showing 10 of {len(classification["nodebags"])} nodebags.</em></p>'
        
        html_content += """
            </div>
            
            <div class="section">
                <h2>Property Table Patterns</h2>
                <p>These patterns represent groups of properties that frequently appear together, suggesting they form a logical table:</p>
        """
        
        # Add property table patterns - avoiding f-string with # issue
        for pattern_name, pattern_info in classification['property_tables'].items():
            html_content += f"""
                <div class="card">
                    <div class="card-title">Pattern: {pattern_name}</div>
                    <p><strong>Properties:</strong> 
            """
            
            # Add property badges
            for p in pattern_info['properties']:
                # Using string operations to avoid f-string with # issue
                property_name = p.split("/")[-1].split("#")[-1]
                html_content += f'<span class="badge">{property_name}</span>'
            
            html_content += f"""
                    </p>
                    <p><strong>Used by {len(pattern_info['entities'])} entities</strong></p>
                </div>
            """
        
        # Final sections of HTML
        html_content += """
            </div>
            
            <div class="section">
                <h2>Schema Diagram Explanation</h2>
                <p>The RDF schema represents a complex graph with the following key elements:</p>
                <ul>
                    <li><strong>Primary Entities:</strong> Main objects in the data model with high connectivity</li>
                    <li><strong>Secondary Entities:</strong> Supporting objects with fewer connections</li>
                    <li><strong>Link Entities:</strong> Objects that primarily serve to connect other entities</li>
                    <li><strong>Nodebags:</strong> Blank nodes that group related properties (similar to embedded documents or complex attributes)</li>
                    <li><strong>Property Tables:</strong> Sets of properties that commonly appear together, suggesting a database table pattern</li>
                </ul>
            </div>
            
            <div class="section">
                <h2>SPARQL Query Patterns</h2>
                <h3>Querying Nodebags:</h3>
                <pre>
SELECT ?entity ?prop1Value ?prop2Value
WHERE {
    ?entity ?reference ?nodebag .
    ?nodebag &lt;property1&gt; ?prop1Value ;
             &lt;property2&gt; ?prop2Value .
}
                </pre>
                
                <h3>Finding Primary Entities with their Related Secondary Entities:</h3>
                <pre>
SELECT ?primary ?secondary ?relationship
WHERE {
    ?primary a &lt;PrimaryClass&gt; .
    ?secondary a &lt;SecondaryClass&gt; .
    ?primary ?relationship ?secondary .
}
                </pre>
            </div>
        </body>
        </html>
        """
        
        # Write the complete HTML content
        f.write(html_content)
    
    print(f"Schema analysis complete! Reports saved to {output_dir}")
    return {
        'text_report': text_report,
        'json_data': json_file,
        'html_report': html_report
    }

def analyze_complex_rdf(file_path=None, graph=None):
    """Generate comprehensive schema analysis for a complex RDF graph."""
    if graph is None and file_path is not None:
        # Load graph from file
        graph = Graph()
        graph.parse(file_path)
        print(f"Loaded {len(graph)} triples from {file_path}")
    
    if graph is None:
        raise ValueError("Either file_path or graph must be provided")
    
    # Generate schema reports
    return generate_schema_report(graph)

In [25]:
analyze_complex_rdf(file_path='./mappings/NKB_RDF_V3.ttl')

Loaded 1369675 triples from ./mappings/NKB_RDF_V3.ttl
Schema analysis complete! Reports saved to complex_rdf_schema


{'text_report': 'complex_rdf_schema/schema_report.txt',
 'json_data': 'complex_rdf_schema/schema_data.json',
 'html_report': 'complex_rdf_schema/schema_report.html'}